# V259 전력사용량 예측 파이프라인

In [1]:
# 설치 → 런타임 재시작 → 다음 셀부터 실행
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.1.3', 'pandas==2.2.3',
                       'scipy==1.16.3', 'scikit-learn==1.6.1', 'threadpoolctl==3.7.0', 'cloudpickle'])
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. 데이터

- 손상 행 제외, 기간 분할(튜닝 1~8월 / 최종 학습 전체)

In [2]:
from collections import Counter
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import json
import pickle

import cloudpickle
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVR

SEED = 42
np.random.seed(SEED)
ROOT = Path('/content/drive/MyDrive/kamp_2026')
DATA_PATH = ROOT / 'data/okm_augumented_2021.csv'
INPUTS = ['날짜', '시간', '생산량', '기온']
POWER = ['15분', '30분', '45분', '60분']
VALIDATION_MONTHS = [202104, 202105, 202106, 202107, 202108]

raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
# 손상 행(시간 ∉ 0~23) 제외
full_df = raw.loc[raw['시간'].between(0, 23), INPUTS + POWER].sort_values(['날짜', '시간']).copy()
# 튜닝(CV) 1~8월, 최종 학습 9월까지 전체
train_df = full_df.loc[full_df['날짜'].between(20210101, 20210831)].copy()
RUN_DIR = ROOT / 'models' / ('v259_' + datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S'))
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('튜닝 행', len(train_df), '/ 최종 학습 행', len(full_df), '/ 저장 위치', RUN_DIR)

튜닝 행: 5784 / 최종 학습 행: 6120 / 저장 위치: /content/drive/MyDrive/kamp_2026/models/v259_20261005_151309


## 2. 피처

- 로그 변환, 생산량×시간·요일 상호작용, 힌지(생산량 100 초과분), sin·cos 시간 인코딩, 제한 날짜 추세, 가동 상태×시간×주말

In [3]:
def metrics(y, pred):
    y, pred = np.asarray(y).ravel(), np.asarray(pred).ravel()
    return dict(MAE=mean_absolute_error(y, pred), RMSE=np.sqrt(mean_squared_error(y, pred)),
                R2=r2_score(y, pred))

def weekday_of(frame):
    return pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek

def is_weekend(frame):
    return weekday_of(frame).ge(5)

def base_features(frame):
    """시간×요일 범주, 기온, 로그 생산량, 시간·요일별 생산량 상호작용"""
    hour = frame['시간']
    weekday = weekday_of(frame)
    log_prod = np.log1p(frame['생산량'])
    on = log_prod.gt(0).where(log_prod.notna()).astype(float)  # 가동 여부
    hinge = (log_prod - np.log1p(100)).clip(lower=0)  # 생산량 100 초과분
    x = pd.DataFrame({
        'hour_weekday': hour.astype(str) + '_' + weekday.astype(str),
        '기온': frame['기온'],
        'log_prod': log_prod,
        'on': on,
        'temp_on': frame['기온'] * on,
        'temp_above20': (frame['기온'] - 20).clip(lower=0),
    }, index=frame.index)
    for day in range(7):
        x[f'prod_day{day}'] = log_prod * weekday.eq(day)
    for h in range(24):
        x[f'prod_hour{h}'] = log_prod * hour.eq(h)
    for h in range(24):
        x[f'prod_hinge_hour{h}'] = hinge * hour.eq(h)
    x['prod_sin'] = log_prod * np.sin(2 * np.pi * hour / 24)
    x['prod_cos'] = log_prod * np.cos(2 * np.pi * hour / 24)
    return x

class HuberFeatures(TransformerMixin, BaseEstimator):
    """본체 Huber 피처. 날짜 추세 기준일 = fit 시 학습 데이터 마지막 날"""

    def fit(self, frame, y=None):
        date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
        self.anchor_ = float(date.max().toordinal())
        return self

    def transform(self, frame):
        x = base_features(frame)
        date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
        ordinal = date.map(lambda value: value.toordinal()).astype(float)
        # 평일 비가동 시간 날짜 추세(최근 60일, -1~0 제한)
        gate = frame['생산량'].eq(0) & date.dt.dayofweek.lt(5)
        x['bounded_date_trend'] = ((ordinal - self.anchor_) / 60).clip(-1, 0) * gate
        # 가동 여부 × 시간 × 평일/주말
        weekend = date.dt.dayofweek.ge(5).astype(int)
        log_prod = np.log1p(frame['생산량'])
        on = log_prod.gt(0).where(log_prod.notna()).astype(float)
        state = pd.DataFrame({
            f'state_hour_{hour}_group_{group}': on * frame['시간'].eq(hour) * weekend.eq(group)
            for hour in range(24) for group in range(2)
        }, index=frame.index)
        x = pd.concat([x, state], axis=1)
        # 기온 × 생산 강도
        x['temperature_intensity'] = frame['기온'] * log_prod
        x['cooling_intensity'] = (frame['기온'] - 20).clip(lower=0) * log_prod
        return x

def residual_features(frame):
    """주말 잔차 보정용: 시간·요일(범주), 기온, 로그 생산량"""
    return pd.DataFrame({
        '시간': frame['시간'],
        '기온': frame['기온'],
        'weekday': weekday_of(frame),
        'log_prod': np.log1p(frame['생산량']),
    }, index=frame.index)

## 3. 모델

- 원-핫 인코딩, 중앙값 대치, 표준화, Huber 회귀(표본 가중치), 주말 잔차 SVR(RBF·poly), 가중 앙상블(0.75·0.25)

In [4]:
huber_pipeline = Pipeline([
    ('features', HuberFeatures()),
    ('preprocess', ColumnTransformer(
        [('category', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False),
          ['hour_weekday'])],
        remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()),
    )),
    ('huber', MultiOutputRegressor(HuberRegressor(max_iter=2000, tol=1e-7))),
])

residual_pipeline = Pipeline([
    ('features', FunctionTransformer(residual_features)),
    ('preprocess', ColumnTransformer(
        [('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday'])],
        remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()),
    )),
    ('svr', MultiOutputRegressor(SVR(coef0=1, tol=1e-4))),
])


def sample_weight(train):
    # 동일 하루 전력 곡선 반복일 가중치: 반복 수^-0.5
    curves = {}
    for date, day in train.groupby('날짜'):
        day = day.sort_values('시간')
        if len(day) == 24 and np.array_equal(day['시간'].to_numpy(), np.arange(24)):
            curves[date] = day[POWER].to_numpy(dtype='<f8').tobytes()
    counts = Counter(curves.values())
    duplicate = train['날짜'].map({date: counts[curve] ** -0.5 for date, curve in curves.items()}).fillna(1.0).to_numpy()
    # 가동 여부×기온 5℃ 구간 가중치: 빈도^-0.5
    group = train['생산량'].eq(0).astype(str) + '_' + train['기온'].floordiv(5).astype('Int64').astype(str)
    support = 1.0 / np.sqrt(group.map(group.value_counts()))
    thermal = (support / support.mean()).to_numpy()
    weight = np.where(train['생산량'].eq(0), .5, 1.) * thermal * duplicate
    return weight / weight.mean()


class V259Model:
    def __init__(self, huber_alpha=1.5, huber_epsilon=1.05,
                 rbf_c=10, rbf_epsilon=1, rbf_gamma='scale',
                 poly_c=10, poly_epsilon=1, poly_degree=2):
        self.huber_alpha, self.huber_epsilon = huber_alpha, huber_epsilon
        self.rbf_c, self.rbf_epsilon, self.rbf_gamma = rbf_c, rbf_epsilon, rbf_gamma
        self.poly_c, self.poly_epsilon, self.poly_degree = poly_c, poly_epsilon, poly_degree

    def fit(self, train):
        inputs = train[INPUTS]
        self.base = clone(huber_pipeline).set_params(
            huber__estimator__alpha=self.huber_alpha, huber__estimator__epsilon=self.huber_epsilon)
        self.base.fit(inputs, train[POWER], huber__sample_weight=sample_weight(train))
        residual = train[POWER].to_numpy() - self.base.predict(inputs)
        weekend = is_weekend(inputs).to_numpy()
        self.residuals = []
        for kernel, numeric_weight, params in [
            ('rbf', .5, dict(C=self.rbf_c, epsilon=self.rbf_epsilon, gamma=self.rbf_gamma, degree=2)),
            ('poly', 1., dict(C=self.poly_c, epsilon=self.poly_epsilon, gamma='scale', degree=self.poly_degree)),
        ]:
            model = clone(residual_pipeline).set_params(
                preprocess__transformer_weights={'remainder': numeric_weight},
                svr__estimator__kernel=kernel,
                **{f'svr__estimator__{key}': value for key, value in params.items()},
            )
            self.residuals.append(model.fit(inputs.loc[weekend], residual[weekend]))
        return self

    def predict(self, frame):
        inputs = frame[INPUTS]
        prediction = self.base.predict(inputs)
        weekend = is_weekend(inputs).to_numpy()
        if weekend.any():
            rbf = self.residuals[0].predict(inputs.loc[weekend])
            poly = self.residuals[1].predict(inputs.loc[weekend])
            prediction[weekend] += .75 * rbf + .25 * poly
        return prediction

## 4. 하이퍼파라미터 튜닝

- 순차 그리드 탐색(Huber → RBF-SVR → poly-SVR), 월별 확장 윈도우 CV(1~3월→4월 … 1~7월→8월), MAE

In [5]:
def run_monthly_cv(params):
    scores, predictions = [], []
    for month in VALIDATION_MONTHS:
        tr = train_df.loc[train_df['날짜'].floordiv(100).lt(month)]
        va = train_df.loc[train_df['날짜'].floordiv(100).eq(month)]
        prediction = V259Model(**params).fit(tr).predict(va)
        scores.append(dict(month=month, **metrics(va[POWER], prediction)))
        predictions.append(pd.DataFrame(prediction, index=va.index, columns=POWER))
    return pd.DataFrame(scores).set_index('month'), pd.concat(predictions)


def tune(name, params, grid):
    best_params, best_mae = params, np.inf
    for candidate in ParameterGrid(grid):
        spec = {**params, **candidate}
        scores, _ = run_monthly_cv(spec)
        mae = scores['MAE'].mean()
        tuning_rows.append({'component': name, **candidate, 'mean_MAE': mae,
                            **{f'MAE_{m}': v for m, v in scores['MAE'].items()}})
        print(name, candidate, f'MAE {mae:.6f}', flush=True)
        if mae < best_mae:
            best_params, best_mae = spec, mae
    print(name, '선택', {key: best_params[key] for key in grid}, f'/ MAE {best_mae:.6f}')
    return best_params


TUNING_SPACES = {
    'Huber': {'huber_alpha': [.5, 1.5, 5.], 'huber_epsilon': [1.05, 1.2, 1.35]},
    'RBF-SVR': {'rbf_c': [3, 10, 30], 'rbf_epsilon': [.5, 1., 2.], 'rbf_gamma': ['scale', .1]},
    'poly-SVR': {'poly_c': [3, 10, 30], 'poly_epsilon': [.5, 1., 2.], 'poly_degree': [2, 3]},
}
params = {}  # 시작값: V259Model 기본값
tuning_rows = []

In [6]:
params = tune('Huber', params, TUNING_SPACES['Huber'])

Huber {'huber_alpha': 0.5, 'huber_epsilon': 1.05} MAE=12.554867
Huber {'huber_alpha': 0.5, 'huber_epsilon': 1.2} MAE=12.606727
Huber {'huber_alpha': 0.5, 'huber_epsilon': 1.35} MAE=12.668380
Huber {'huber_alpha': 1.5, 'huber_epsilon': 1.05} MAE=12.297637
Huber {'huber_alpha': 1.5, 'huber_epsilon': 1.2} MAE=12.321782
Huber {'huber_alpha': 1.5, 'huber_epsilon': 1.35} MAE=12.327960
Huber {'huber_alpha': 5.0, 'huber_epsilon': 1.05} MAE=12.027309
Huber {'huber_alpha': 5.0, 'huber_epsilon': 1.2} MAE=12.067251
Huber {'huber_alpha': 5.0, 'huber_epsilon': 1.35} MAE=12.139342
Huber 선택: {'huber_alpha': 5.0, 'huber_epsilon': 1.05} / MAE=12.027309


In [7]:
params = tune('RBF-SVR', params, TUNING_SPACES['RBF-SVR'])

RBF-SVR {'rbf_c': 3, 'rbf_epsilon': 0.5, 'rbf_gamma': 'scale'} MAE=12.051763
RBF-SVR {'rbf_c': 3, 'rbf_epsilon': 0.5, 'rbf_gamma': 0.1} MAE=12.124042
RBF-SVR {'rbf_c': 3, 'rbf_epsilon': 1.0, 'rbf_gamma': 'scale'} MAE=12.056758
RBF-SVR {'rbf_c': 3, 'rbf_epsilon': 1.0, 'rbf_gamma': 0.1} MAE=12.128520
RBF-SVR {'rbf_c': 3, 'rbf_epsilon': 2.0, 'rbf_gamma': 'scale'} MAE=12.088568
RBF-SVR {'rbf_c': 3, 'rbf_epsilon': 2.0, 'rbf_gamma': 0.1} MAE=12.141022
RBF-SVR {'rbf_c': 10, 'rbf_epsilon': 0.5, 'rbf_gamma': 'scale'} MAE=12.025778
RBF-SVR {'rbf_c': 10, 'rbf_epsilon': 0.5, 'rbf_gamma': 0.1} MAE=12.139894
RBF-SVR {'rbf_c': 10, 'rbf_epsilon': 1.0, 'rbf_gamma': 'scale'} MAE=12.027309
RBF-SVR {'rbf_c': 10, 'rbf_epsilon': 1.0, 'rbf_gamma': 0.1} MAE=12.141396
RBF-SVR {'rbf_c': 10, 'rbf_epsilon': 2.0, 'rbf_gamma': 'scale'} MAE=12.062770
RBF-SVR {'rbf_c': 10, 'rbf_epsilon': 2.0, 'rbf_gamma': 0.1} MAE=12.164315
RBF-SVR {'rbf_c': 30, 'rbf_epsilon': 0.5, 'rbf_gamma': 'scale'} MAE=12.049693
RBF-SVR {'rbf_c'

In [8]:
params = tune('poly-SVR', params, TUNING_SPACES['poly-SVR'])

poly-SVR {'poly_c': 3, 'poly_degree': 2, 'poly_epsilon': 0.5} MAE=12.020426
poly-SVR {'poly_c': 3, 'poly_degree': 2, 'poly_epsilon': 1.0} MAE=12.019667
poly-SVR {'poly_c': 3, 'poly_degree': 2, 'poly_epsilon': 2.0} MAE=12.026365
poly-SVR {'poly_c': 3, 'poly_degree': 3, 'poly_epsilon': 0.5} MAE=11.987275
poly-SVR {'poly_c': 3, 'poly_degree': 3, 'poly_epsilon': 1.0} MAE=11.987412
poly-SVR {'poly_c': 3, 'poly_degree': 3, 'poly_epsilon': 2.0} MAE=11.988945
poly-SVR {'poly_c': 10, 'poly_degree': 2, 'poly_epsilon': 0.5} MAE=12.026275
poly-SVR {'poly_c': 10, 'poly_degree': 2, 'poly_epsilon': 1.0} MAE=12.025778
poly-SVR {'poly_c': 10, 'poly_degree': 2, 'poly_epsilon': 2.0} MAE=12.031206
poly-SVR {'poly_c': 10, 'poly_degree': 3, 'poly_epsilon': 0.5} MAE=11.994108
poly-SVR {'poly_c': 10, 'poly_degree': 3, 'poly_epsilon': 1.0} MAE=11.994956
poly-SVR {'poly_c': 10, 'poly_degree': 3, 'poly_epsilon': 2.0} MAE=12.003420
poly-SVR {'poly_c': 30, 'poly_degree': 2, 'poly_epsilon': 0.5} MAE=12.022554
poly-

In [9]:
# 선택 설정 월별 CV 결과·검증 예측(OOF) 저장
cv_scores, oof = run_monthly_cv(params)
tuning_results = pd.DataFrame(tuning_rows)
tuning_results.to_csv(RUN_DIR / 'tuning_results.csv', index=False)
cv_scores.to_csv(RUN_DIR / 'V259_cv.csv')
train_df.loc[oof.index, ['날짜', '시간'] + POWER].join(oof.add_prefix('pred_')).to_csv(
    RUN_DIR / 'V259_oof.csv', index_label='row_id')
(RUN_DIR / 'params.json').write_text(json.dumps(params, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.Series(params, name='selected'))
display(cv_scores)
print('월평균 CV MAE', cv_scores['MAE'].mean())

,selected
huber_alpha,5.0
huber_epsilon,1.05
rbf_c,10
rbf_epsilon,0.5
rbf_gamma,scale
poly_c,3
poly_degree,3
poly_epsilon,0.5


,MAE,RMSE,R2
month,,,
202104,15.827794,26.629061,0.756281
202105,11.252055,23.807280,0.811815
202106,11.836302,20.118836,0.868978
202107,12.785670,23.634501,0.850164
202108,8.234555,15.204615,0.939437


월평균 CV MAE: 11.987274935163224


## 5. 전체 데이터 재학습·저장

- 전체 데이터 재학습, cloudpickle 직렬화

In [10]:
model = V259Model(**params).fit(full_df)
bundle = {'model': model, 'params': params, 'input_columns': INPUTS, 'target_columns': POWER,
          'cv_scores': cv_scores, 'oof': oof, 'tuning_results': tuning_results}
PKL_PATH = RUN_DIR / 'v259_full_data.pkl'
with PKL_PATH.open('wb') as file:
    cloudpickle.dump(bundle, file)
print('저장', PKL_PATH)

저장: /content/drive/MyDrive/kamp_2026/models/v259_20261005_151309/v259_full_data.pkl


## 6. 추론

- 저장 모델 로드 후 예측

In [11]:
# 입력 날짜·시간·생산량·기온 → 출력 15분·30분·45분·60분 전력
with PKL_PATH.open('rb') as file:
    loaded = pickle.load(file)
new_inputs = full_df[INPUTS].tail(24)
predictions = pd.DataFrame(loaded['model'].predict(new_inputs), index=new_inputs.index,
                           columns=loaded['target_columns'])
display(new_inputs.join(predictions.add_prefix('pred_')))

,날짜,시간,생산량,기온,pred_15분,pred_30분,pred_45분,pred_60분
6144,20210914,0,128,21.9,72.258659,69.396989,68.268380,67.962596
6145,20210914,1,45,21.8,92.502055,102.732156,106.930351,106.576428
6146,20210914,2,479,22.2,109.274476,108.946450,107.632251,108.745004
6147,20210914,3,93,22.5,93.244161,108.056182,109.917921,107.697481
6148,20210914,4,255,21.1,106.412573,106.130407,104.871595,104.549165
6149,20210914,5,76,21.5,81.072423,73.286638,94.713251,105.976611
6150,20210914,6,1228,21.3,112.127732,108.503315,105.113999,102.289069
6151,20210914,7,1110,21.7,105.940443,115.746661,141.772593,141.274986
6152,20210914,8,432,21.8,158.374153,173.981253,175.273296,175.279356
6153,20210914,9,1672,21.3,173.169664,173.092516,174.411889,177.548318
